In [ ]:
# Cell 132 (04_baselines_rq2.ipynb, cell 1): RUN AFTER A KERNEL RESTART, ONLY THIS CELL. Read-only, writes NOTHING.
def _c132():
    import os, re, json, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    import pgmpy, sklearn, numpy, pandas
    print("pgmpy", pgmpy.__version__, "| sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)

    new = ["selection/tables/sel_baselines_table.csv", "selection/tables/sel_baselines_bootstrap.csv", "selection/tables/sel_em_vs_deletion.csv",
           "selection/tables/sel_rq2_sensitivity_table.csv", "selection/predictions/sel_rq2_per_row.pkl", "selection/tables/sel_voi_partA_table.csv",
           "selection/tables/sel_voi_partB_table.csv", "selection/tables/sel_cost_ratio_grid.csv", "shared/src/sensitivity.py"]
    print("\nname collisions:", [n for n in new if os.path.exists(os.path.join(R, n))] or "none")
    print("inputs:", {n: os.path.exists(os.path.join(R, n)) for n in ["selection/predictions/sel_bn_test_predictions.pkl", "selection/models/sel_bn_final.pkl",
          "selection/notebooks/00_precheck.ipynb", "selection/predictions/sel_precheck_predictions.pkl", "Data/processed/sel_test.pkl"]})

    def cells(path):
        nb = json.load(open(path)); out = []
        for ci, c in enumerate(nb["cells"]):
            if c.get("cell_type") == "code":
                out.append((ci, c.get("execution_count"), "".join(c["source"]) if isinstance(c["source"], list) else c["source"]))
        return out

    print("\n================ 00_precheck.ipynb: code cells (Phase 1 pipeline) ================")
    shown = 0
    for ci, ec, s in cells(os.path.join(R, "selection/notebooks/00_precheck.ipynb")):
        print(f"\n--- cell {ci} [exec {ec}] ({len(s.splitlines())} lines) ---")
        lines = s.splitlines(); print("\n".join(lines[:90]))
        if len(lines) > 90: print("   ... (%d more lines)" % (len(lines) - 90))
        shown += 1

    fz = cells(os.path.join(R, "full_variable/notebooks/Expirements_FROZEN.ipynb"))
    print("\n================ frozen notebook: grid / CV / imputer lines ================")
    pat = re.compile(r"GridSearchCV|param_grid|best_params|best_score|cv\s*=|StratifiedKFold|scoring|n_jobs|IterativeImputer|SimpleImputer|KNNImputer|class_weight|max_iter|solver|random_state|RandomForestClassifier\(|LogisticRegression\(", re.I)
    for ci, ec, s in fz:
        hits = [l.strip()[:190] for l in s.splitlines() if pat.search(l) and not l.strip().startswith("#")]
        if hits:
            print(f"cell {ci} [exec {ec}]")
            for h in hits[:14]: print("    ", h)

    rk = open(os.path.join(R, "shared/src/_recovered_from_kernel.py")).read().splitlines()
    starts = {m.group(1): i for i, l in enumerate(rk) for m in [re.match(r"^def (\w+)\(", l)] if m}
    def body(n):
        i = starts[n]; j = i + 1
        while j < len(rk) and (rk[j].strip() == "" or rk[j][0] in " \t"): j += 1
        return "\n".join(rk[i:j])
    print("\n================ recovered kernel: sensitivity / bootstrap functions ================")
    for n in ["posterior_default", "make_posterior_fn", "post_B", "p_default0", "row_sensitivity", "max_shift_for", "support", "zero_nodes",
              "analyze_blanket", "summarize", "summary_table", "stat_row", "paired_bootstrap", "bootstrap_auc_diff", "compute_ks", "eval_probs"]:
        if n in starts:
            b = body(n).splitlines(); print(f"\n--- {n} ({len(b)} lines) ---"); print("\n".join(b[:60]))
            if len(b) > 60: print("   ... (%d more lines)" % (len(b) - 60))
        else:
            print(f"\n--- {n}: not found ---")

    print("\n================ frozen cells 63-66, 70, 76 (Part A / VoI loops), first 70 lines each ================")
    for ci, ec, s in fz:
        if ci in (63, 64, 65, 70, 76):
            lines = s.splitlines(); print(f"\n--- cell {ci} [exec {ec}] ({len(lines)} lines) ---"); print("\n".join(lines[:70]))
            if len(lines) > 70: print("   ... (%d more lines)" % (len(lines) - 70))
_c132()

In [ ]:
# Cell 132b (04_baselines_rq2.ipynb): locked-figure style extraction. Read-only, writes NOTHING.
def _c132b():
    import os, re, json, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    fv = os.path.join(R, "full_variable/figures")
    print("locked figures:")
    for f in sorted(os.listdir(fv)):
        print("  ", f, os.path.getsize(os.path.join(fv, f)))
    nb = json.load(open(os.path.join(R, "full_variable/notebooks/Expirements_FROZEN.ipynb")))
    pat = re.compile(r"rcParams|figsize|plt\.style|sns\.set|set_theme|fontsize|fontweight|font\.|color\s*=|cmap|palette|set_xlabel|set_ylabel|set_title|suptitle|savefig|dpi|tab:|#[0-9a-fA-F]{6}|\.legend\(|grid\(", re.I)
    for ci, c in enumerate(nb["cells"]):
        if c.get("cell_type") != "code":
            continue
        s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        if "savefig" not in s:
            continue
        hits = [l.strip()[:200] for l in s.splitlines() if pat.search(l) and not l.strip().startswith("#")]
        print(f"\n--- cell {ci} [exec {c.get('execution_count')}] ({len(s.splitlines())} lines) ---")
        for h in hits[:30]:
            print("   ", h)
_c132b()

In [ ]:
# Cell 132c (04_baselines_rq2.ipynb): grids + remaining locked sensitivity source. Read-only, writes NOTHING.
def _c132c():
    import os, json, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    nb = json.load(open(os.path.join(R, "full_variable/notebooks/Expirements_FROZEN.ipynb")))
    src = lambda ci: "".join(nb["cells"][ci]["source"]) if isinstance(nb["cells"][ci]["source"], list) else nb["cells"][ci]["source"]
    for ci in (39, 42):
        print(f"\n===== frozen cell {ci} (full) =====\n{src(ci)}")
    for ci, a in ((70, 60), (76, 60)):
        lines = src(ci).splitlines()
        print(f"\n===== frozen cell {ci}, lines {a+1}-{len(lines)} =====")
        print("\n".join(lines[a:]))
    for ci in (66, 67, 71, 72, 73, 74, 77, 78):
        lines = src(ci).splitlines()
        print(f"\n===== frozen cell {ci} ({len(lines)} lines, first 45) =====")
        print("\n".join(lines[:45]))
_c132c()

In [ ]:
# Cell 133 (04_baselines_rq2.ipynb): candidate shared/src/sensitivity.py + validation on the locked 13-node model. Writes NOTHING. No restart.
SENS_SRC = '''"""Sensitivity and value-of-information loops ported from the locked kernel code (no logic changes).
Notebook globals of the locked code are explicit arguments here. Additions that do not change locked outputs:
analyze_blanket(skip_unobserved=...) and an extra trailing column retracted_prob; boot_means groups the locked bootstrap loop."""
import os
assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED != 0: wrong kernel"
import time
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import roc_auc_score


def make_posterior_fn(model):
    eng = VariableElimination(model)
    cache_m = {}
    def f(evidence):
        key = tuple(sorted(evidence.items(), key=lambda kv: kv[0]))
        if key in cache_m:
            return cache_m[key]
        with np.errstate(invalid='ignore', divide='ignore'):
            try:
                p = eng.query(variables=['Default'], evidence=evidence, show_progress=False).get_value(Default=1)
            except Exception:
                p = np.nan
        if not np.isfinite(p) or not (0 <= p <= 1):
            p = np.nan
        cache_m[key] = p
        return p
    return f


def analyze_blanket(model, rows, evid_all, mb, STATES, tau, skip_unobserved=False, verbose=True):
    post = make_posterior_fn(model)
    eng = VariableElimination(model)
    wcache = {}
    def pred_dist(var, ev_rest):
        key = (var, tuple(sorted(ev_rest.items(), key=lambda kv: kv[0])))
        if key in wcache:
            return wcache[key]
        with np.errstate(invalid='ignore', divide='ignore'):
            try:
                f = eng.query(variables=[var], evidence=ev_rest, show_progress=False)
                vals = np.asarray(f.values, dtype=float)
                w = dict(zip(f.state_names[var], vals)) if np.all(np.isfinite(vals)) else None
            except Exception:
                w = None
        wcache[key] = w
        return w
    recs, t0 = [], time.time()
    for i, idx in enumerate(rows):
        ev = evid_all[idx]
        base = post(ev)
        if np.isnan(base):
            continue
        for var in mb:
            if skip_unobserved and var not in ev:
                continue
            xo = ev[var]
            e_rest = {k: v for k, v in ev.items() if k != var}
            shifts, skipped = {}, 0
            for s in STATES[var]:
                if s == xo:
                    continue
                p2 = post({**ev, var: s})
                if np.isnan(p2):
                    skipped += 1
                else:
                    shifts[s] = abs(p2 - base)
            S_max = max(shifts.values()) if shifts else np.nan
            w = pred_dist(var, e_rest)
            if w is None:
                S_exp, w_obs = np.nan, np.nan
            else:
                S_exp = sum(ws * shifts[s] for s, ws in w.items() if s != xo and ws > 0 and s in shifts)
                w_obs = w.get(xo, np.nan)
            p_r = post(e_rest)
            delta = np.nan if np.isnan(p_r) else abs(p_r - base)
            flip = np.nan if np.isnan(p_r) else int((base >= tau) != (p_r >= tau))
            recs.append((idx, var, S_max, S_exp, delta, flip, base, skipped, w_obs, p_r))
        if verbose and (i + 1) % 10000 == 0:
            print(f"  {i+1}/{len(rows)} rows, {time.time()-t0:.0f}s")
    return pd.DataFrame(recs, columns=['row_index', 'variable', 'S_max', 'S_exp', 'delta', 'flip',
                                       'baseline_prob', 'skipped', 'w_obs', 'retracted_prob'])


def partB_check(post, rows, evid_all, m_var, val_var, STATES, tau):
    rows_idx = [i for i in rows if evid_all[i][m_var] == 0]
    worst, flips, skipped = 0.0, 0, 0
    for idx in rows_idx:
        ev = evid_all[idx]; base = post(ev)
        for s in STATES[val_var]:
            p2 = post({**ev, m_var: 1, val_var: s})
            if np.isnan(p2):
                skipped += 1
                continue
            worst = max(worst, abs(p2 - base))
            flips += int((base >= tau) != (p2 >= tau))
    return len(rows_idx), worst, flips, skipped


def paired_bootstrap(y, pa, pb, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y)
    obs = roc_auc_score(y, pa) - roc_auc_score(y, pb)
    diffs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yb = y[idx]
        if yb.min() == yb.max():
            continue
        diffs.append(roc_auc_score(yb, pa[idx]) - roc_auc_score(yb, pb[idx]))
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = max(2 * min((diffs > 0).mean(), (diffs < 0).mean()), 1.0 / len(diffs))
    return obs, lo, hi, p, len(diffs)


def boot_means(mats, B=2000, seed=42):
    """Row-bootstrap of column means; every matrix in mats uses the same resamples (as in the locked loop)."""
    rng = np.random.RandomState(seed)
    n = len(mats[0])
    out = [np.empty((B, m.shape[1])) for m in mats]
    for i in range(B):
        ix = rng.randint(0, n, n)
        for o, m in zip(out, mats):
            o[i] = m[ix].mean(axis=0)
    return out
'''

def _c133():
    import os, time, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    print("candidate sensitivity.py sha12:", hashlib.sha256(SENS_SRC.encode()).hexdigest()[:12])
    ns = {}; exec(compile(SENS_SRC, "sensitivity_candidate", "exec"), ns)
    from pgmpy.models import DiscreteBayesianNetwork

    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); bn.add_cpds(*d["cpds"]); assert bn.check_model()
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    FEATURE_BINS = ['Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)',
                    '30-59 Days Past Due (bin)', '90+ Days Late (bin)', '60-89 Days Past Due (bin)',
                    'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)']
    for c in FEATURE_BINS + ["M_Income", "M_Dependents", "Default"]:
        assert c in te.columns, "missing column %s: STOP" % c
    E = list(bn.edges())
    parents = {u for u, v in E if v == "Default"}; children = {v for u, v in E if u == "Default"}
    co = {u for u, v in E if v in children and u != "Default"}
    mb = sorted(parents | children | co)
    print("Markov blanket of Default (locked model):", mb)
    assert mb == sorted(['Credit Utilisation Rate (bin)', '90+ Days Late (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)']), "unexpected blanket: STOP"
    STATES = {v: bn.get_cpds(v).state_names[v] for v in FEATURE_BINS}
    evid_all = {}
    for idx, row in te.iterrows():
        ev = {c: row[c] for c in FEATURE_BINS if pd.notnull(row[c])}
        ev["M_Income"] = int(row["M_Income"]); ev["M_Dependents"] = int(row["M_Dependents"])
        evid_all[idx] = ev
    rows_all = list(te.index)
    TAU = 0.17

    t0 = time.time()
    ns["analyze_blanket"](bn, rows_all[:200], evid_all, mb, STATES, TAU, verbose=False)
    per = (time.time() - t0) / 200; proj = per * len(rows_all) / 60
    print("timing: %.1f ms/row on 200 cold rows; projected full run <= %.1f min (cache warms, so likely less)" % (per * 1000, proj))
    if proj > 30:
        print("STOP: projected > 30 min. Report before the full run."); return None

    t0 = time.time()
    res = ns["analyze_blanket"](bn, rows_all, evid_all, mb, STATES, TAU, verbose=True)
    print("full run: %d records in %.0fs" % (len(res), time.time() - t0))
    n_per = res.groupby("variable").size().to_dict()
    skipped_rows = set(rows_all) - set(res.row_index.unique())
    print("records per variable:", n_per, "| rows skipped (NaN posterior):", sorted(skipped_rows))
    ok = all(v == 29998 for v in n_per.values()) and skipped_rows == {125238, 7312}

    pth = os.path.join(R, "full_variable/per_row/vo_blanket_original_per_row.pkl")
    if not os.path.exists(pth):
        print("saved per-row file not found at", pth, "| per_row folder:", sorted(os.listdir(os.path.join(R, "full_variable/per_row")))); return None
    sv = pd.read_pickle(pth)
    m = res.merge(sv, on=["row_index", "variable"], suffixes=("", "_sv"))
    assert len(m) == len(res) == len(sv), "row counts differ from saved file: STOP"
    print("\nrecomputed vs saved locked per-row results (%d records):" % len(m))
    for c in ["S_max", "S_exp", "delta", "flip", "baseline_prob", "w_obs", "skipped"]:
        a, b = m[c].astype(float), m[c + "_sv"].astype(float)
        df_ = float(np.nanmax(np.abs(a - b))); nm = bool((a.isna() == b.isna()).all())
        print("  %-14s max |diff| %.3e | NaN positions match: %s" % (c, df_, nm))
        ok = ok and df_ < 1e-10 and nm

    short = lambda v: v.replace(" (bin)", "")
    TGT_S = {'Credit Utilisation Rate (bin)': 0.0280, '30-59 Days Past Due (bin)': 0.0245, '90+ Days Late (bin)': 0.0223, '60-89 Days Past Due (bin)': 0.0111}
    TGT_F = {'Credit Utilisation Rate (bin)': 1374, '30-59 Days Past Due (bin)': 1390, '90+ Days Late (bin)': 927, '60-89 Days Past Due (bin)': 419}
    print("\nvalidation targets:")
    for v in TGT_S:
        g = res[res.variable == v]; mS = round(float(g.S_exp.mean()), 4); nF = int(g.flip.sum())
        o1, o2 = abs(mS - TGT_S[v]) < 1e-9, nF == TGT_F[v]
        ok = ok and o1 and o2
        print("  %-26s mean S_exp %.4f (target %.4f) %s | flips at 0.17: %d (target %d) %s" % (short(v), mS, TGT_S[v], "OK" if o1 else "MISMATCH", nF, TGT_F[v], "OK" if o2 else "MISMATCH"))

    post = ns["make_posterior_fn"](bn)
    rows_ok = sorted(set(res.row_index.unique()))
    print("\nVoI Part B on the locked model (observe the missing value):")
    for m_var, val_var in [("M_Income", "Monthly Income (bin)"), ("M_Dependents", "Number of Dependents (bin)")]:
        n_, worst, flips, sk = ns["partB_check"](post, rows_ok, evid_all, m_var, val_var, STATES, TAU)
        print("  %-26s %d incomplete rows | max shift %.3e | flips %d | skipped states %d" % (short(val_var), n_, worst, flips, sk))
        ok = ok and worst < 1e-12 and flips == 0

    try:
        pp = pd.read_pickle(os.path.join(R, "selection/predictions/sel_precheck_predictions.pkl"))
        print("\n(info) sel_precheck_predictions.pkl:", type(pp).__name__, list(pp.keys()) if isinstance(pp, dict) else "")
        print(pd.read_csv(os.path.join(R, "selection/tables/sel_precheck_bootstrap.csv")).to_string(index=False))
    except Exception as e:
        print("(info) precheck files not readable:", e)
    print("\nVERDICT:", "PASS" if ok else "MISMATCH: STOP")
    return ok

V133 = _c133()

In [ ]:
# Cell 134 (04_baselines_rq2.ipynb): REAL save of shared/src/sensitivity.py (exclusive create) + re-import check.
def _c134():
    import os, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "SENS_SRC" in globals(), "SENS_SRC not defined: re-run Cell 133 first. STOP"
    want = hashlib.sha256(SENS_SRC.encode()).hexdigest()
    assert want[:12] == "e49b368ee726", "source differs from the validated candidate: STOP"
    p = os.path.join(R, "shared/src/sensitivity.py")
    with open(p, "x", encoding="utf-8") as f:
        f.write(SENS_SRC)
    got = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert got == want, "read-back hash mismatch: STOP"
    os.chmod(p, 0o444)
    print("wrote", p, "| %d bytes | sha256 %s" % (os.path.getsize(p), got[:12]))

    spec = importlib.util.spec_from_file_location("sensitivity_saved", p)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    from pgmpy.models import DiscreteBayesianNetwork
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); bn.add_cpds(*d["cpds"])
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    FB = ['Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', '30-59 Days Past Due (bin)',
          '90+ Days Late (bin)', '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)']
    STATES = {v: bn.get_cpds(v).state_names[v] for v in FB}
    mb = ['30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)', 'Credit Utilisation Rate (bin)']
    ev = {}
    for idx, row in te.iterrows():
        e = {c: row[c] for c in FB if pd.notnull(row[c])}; e["M_Income"] = int(row["M_Income"]); e["M_Dependents"] = int(row["M_Dependents"]); ev[idx] = e
    rows = [i for i in te.index if i not in (7312, 125238)][:300]
    res = mod.analyze_blanket(bn, rows, ev, mb, STATES, 0.17, verbose=False)
    sv = pd.read_pickle(os.path.join(R, "full_variable/per_row/vo_blanket_original_per_row.pkl"))
    m = res.merge(sv, on=["row_index", "variable"], suffixes=("", "_sv"))
    diff = max(float(np.nanmax(np.abs(m[c].astype(float) - m[c + "_sv"].astype(float)))) for c in ["S_max", "S_exp", "delta", "flip", "baseline_prob"])
    print("from disk, 300-row slice (%d records): max |diff| vs saved locked results = %.3e" % (len(m), diff))
    print("VERDICT:", "PASS" if diff < 1e-10 and len(m) == 1200 else "MISMATCH")

_c134()

In [ ]:
# Cell 135 (04_baselines_rq2.ipynb): Part A preflight. Rebuild Spec A design, sanity check vs Phase 1, time fits. Writes NOTHING. No restart.
def _c135():
    import os, json, time, pickle, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    PROC = R + "/Data/processed/"
    L = lambda p: pickle.load(open(p, "rb"))

    nb = json.load(open(R + "/selection/notebooks/00_precheck.ipynb"))
    src = lambda ci: "".join(nb["cells"][ci]["source"])
    g = {"os": os, "np": np, "pd": pd, "L": L, "PROC": PROC, "ROOT": R}
    split = L(PROC + "split_indices.pkl")
    g["train_idx"], g["test_idx"] = np.asarray(split["train_idx"]), np.asarray(split["test_idx"])
    g["y_train"], g["y_test"] = np.asarray(split["y_train"]), np.asarray(split["y_test"])
    code = src(2)
    cut = code.index("# ---- validation")
    exec(compile(code[:cut], "build_design_src", "exec"), g)      # defines build_design and helpers only
    build_design = g["build_design"]
    SPEC_A = ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents']
    t0 = time.time(); D = build_design(SPEC_A)
    print("Spec A design: train %s, test %s, built in %.0fs" % (D["Xtr"].shape, D["Xte"].shape, time.time() - t0))
    print("columns:", list(D["Xtr"].columns))
    y_tr, y_te = g["y_train"], g["y_test"]

    lr = LogisticRegression(C=100, penalty="l2", solver="saga", class_weight="balanced", max_iter=1000, random_state=42)
    t0 = time.time(); lr.fit(D["Xtr"].values, y_tr); t_lr = time.time() - t0
    p = lr.predict_proba(D["Xte"].values)[:, 1]
    pp = L(R + "/selection/predictions/sel_precheck_predictions.pkl")
    print("LR (C=100, l2) fit %.1fs | test AUC %.6f | Phase 1 Spec A AUC %.6f | max |p - phase1| %.3e" % (
        t_lr, roc_auc_score(y_te, p), roc_auc_score(y_te, pp["A"]), np.max(np.abs(p - np.asarray(pp["A"])))))

    rf = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=10, class_weight="balanced", random_state=42, n_jobs=1)
    t0 = time.time(); rf.fit(D["Xtr"].values, y_tr); t_rf = time.time() - t0
    print("RF (500/15/10, n_jobs=1) fit %.1fs | test AUC %.4f" % (t_rf, roc_auc_score(y_te, rf.predict_proba(D["Xte"].values)[:, 1])))

    nbf = json.load(open(R + "/full_variable/notebooks/Expirements_FROZEN.ipynb"))
    fs = lambda ci: "".join(nbf["cells"][ci]["source"])
    assert "'C': [0.01, 0.1, 1, 10, 100]" in fs(39) and "'penalty': ['l1', 'l2']" in fs(39)
    assert "'n_estimators': [100, 200, 500]" in fs(42) and "'max_depth': [5, 10, 15]" in fs(42) and "'min_samples_leaf': [5, 10, 20]" in fs(42)
    print("\nfrozen grids confirmed: LR C x penalty = 10 combos; RF trees x depth x leaf = 27 combos (restricted grid)")
    print("projected serial time: LR grid ~%.0f min (upper bound, l1 is slower) | RF grid ~%.0f min (trees scale 100:200:500, avg ~%.1fs/fit x 135 fits)" % (
        t_lr * 50 / 60, t_rf * (800 / 500) / 3 * 135 / 60 * 0 + (t_rf * (100 + 200 + 500) / 500 / 3) * 45 / 60, t_rf * 800 / 500 / 3))
    print("pre-flight:", "PASS" if abs(roc_auc_score(y_te, p) - roc_auc_score(y_te, pp["A"])) < 1e-9 else "CHECK: LR does not reproduce Phase 1")
_c135()

In [ ]:
# Cell 136 (04_baselines_rq2.ipynb): Part A grid searches (serial, ~45 min). Saves sel_lr_grid.pkl / sel_rf_grid.pkl (exclusive create). Resumable. No restart.
def get_design():
    import os, json, pickle, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    PROC = R + "/Data/processed/"
    L = lambda p: pickle.load(open(p, "rb"))
    nb = json.load(open(R + "/selection/notebooks/00_precheck.ipynb"))
    code = "".join(nb["cells"][2]["source"])
    g = {"os": os, "np": np, "pd": pd, "L": L, "PROC": PROC, "ROOT": R}
    split = L(PROC + "split_indices.pkl")
    g["train_idx"], g["test_idx"] = np.asarray(split["train_idx"]), np.asarray(split["test_idx"])
    g["y_train"], g["y_test"] = np.asarray(split["y_train"]), np.asarray(split["y_test"])
    exec(compile(code[:code.index("# ---- validation")], "build_design_src", "exec"), g)
    D = g["build_design"](['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents'])
    return D, g["y_train"], g["y_test"]

def _c136():
    import os, sys, time, pickle, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    D, y_tr, y_te = get_design()
    X = D["Xtr"].values
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    specs = {
        "lr": ("selection/models/sel_lr_grid.pkl",
               LogisticRegression(class_weight="balanced", solver="saga", max_iter=1000, random_state=42),
               {"C": [0.01, 0.1, 1, 10, 100], "penalty": ["l1", "l2"]}),
        "rf": ("selection/models/sel_rf_grid.pkl",
               RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1),
               {"n_estimators": [100, 200, 500], "max_depth": [5, 10, 15], "min_samples_leaf": [5, 10, 20]}),
    }
    out = {}
    for k, (rel, base, grid) in specs.items():
        p = os.path.join(R, rel)
        if os.path.exists(p):
            out[k] = pickle.load(open(p, "rb")); print("%s: found saved grid, skipping fit" % k); continue
        ss.check_path(p)
        t0 = time.time()
        gs = GridSearchCV(base, grid, scoring="roc_auc", cv=cv, n_jobs=1, verbose=0).fit(X, y_tr)
        print("%s grid done in %.1f min" % (k, (time.time() - t0) / 60))
        blob = pickle.dumps(gs)
        with open(ss.check_path(p), "xb") as f: f.write(blob)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(blob).hexdigest()
        print("saved %s (sha12 %s)" % (rel, hashlib.sha256(blob).hexdigest()[:12]))
        out[k] = gs
    for k, (rel, base, grid) in specs.items():
        gs = out[k]; cvr = pd.DataFrame(gs.cv_results_)
        print("\n=== %s ===\nbest_params_: %s | best CV AUC: %.4f (std over folds %.4f)" % (
            k.upper(), gs.best_params_, gs.best_score_, cvr.loc[gs.best_index_, "std_test_score"]))
        cols = ["param_" + n for n in grid] + ["mean_test_score", "std_test_score"]
        print("top 5 by CV AUC:"); print(cvr.sort_values("mean_test_score", ascending=False)[cols].head(5).to_string(index=False, float_format=lambda x: "%.4f" % x))
        edge = {n: ("min edge" if gs.best_params_[n] == v[0] else "max edge" if gs.best_params_[n] == v[-1] else "interior")
                for n, v in grid.items() if n != "penalty"}
        print("grid-edge status of the chosen values:", edge, "| penalty chosen:", gs.best_params_.get("penalty", "n/a"))
    print("\nlocked full-variable choices for reference: LR C=100 l2 (CV 0.8582); RF 500/15/10 (CV 0.8559)")
_c136()

In [ ]:
# Cell 137 (04_baselines_rq2.ipynb): Part A tests. OOF thresholds, test metrics, paired bootstrap, additive diagnostic. No restart.
def _c137():
    import os, sys, time, pickle, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.base import clone
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "get_design" in globals(), "get_design not defined: re-run the Cell 136 definitions first. STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    O_T = "selection/tables/sel_baselines_table.csv"; O_B = "selection/tables/sel_baselines_bootstrap.csv"
    O_P = "selection/predictions/sel_baseline_test_predictions.pkl"
    for rel in (O_T, O_B, O_P): ss.check_path(os.path.join(R, rel))

    D, y_tr, y_te = get_design()
    Xtr, Xte = D["Xtr"].values, D["Xte"].values
    lr_gs = pickle.load(open(R + "/selection/models/sel_lr_grid.pkl", "rb"))
    rf_gs = pickle.load(open(R + "/selection/models/sel_rf_grid.pkl", "rb"))
    lr, rf = lr_gs.best_estimator_, rf_gs.best_estimator_
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # --- LR convergence diagnostics (descriptive) ---
    cvr = pd.DataFrame(lr_gs.cv_results_)
    print("LR best_estimator_: C=%s penalty=%s | n_iter_=%s (max_iter=%d)" % (lr.C, lr.penalty, lr.n_iter_, lr.max_iter))
    print("CV AUC of C=100 l2 in the grid: %.4f | of C=1 l2: %.4f" % (
        cvr[(cvr.param_C == 100) & (cvr.param_penalty == "l2")].mean_test_score.iloc[0],
        cvr[(cvr.param_C == 1) & (cvr.param_penalty == "l2")].mean_test_score.iloc[0]))
    for C in (1, 100):
        m = LogisticRegression(C=C, penalty="l2", solver="saga", class_weight="balanced", max_iter=1000, random_state=42).fit(Xtr, y_tr)
        print("  saga C=%-3d n_iter_ %s | test AUC %.4f" % (C, m.n_iter_, roc_auc_score(y_te, m.predict_proba(Xte)[:, 1])))
    for C in (1, 100):
        m = LogisticRegression(C=C, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(Xtr, y_tr)
        print("  lbfgs C=%-3d n_iter_ %s | test AUC %.4f   (diagnostic only)" % (C, m.n_iter_, roc_auc_score(y_te, m.predict_proba(Xte)[:, 1])))

    # --- predictions ---
    p_lr = lr.predict_proba(Xte)[:, 1]; p_rf = rf.predict_proba(Xte)[:, 1]
    b = pickle.load(open(R + "/selection/predictions/sel_bn_test_predictions.pkl", "rb"))
    assert bool(np.all(b["included"])) and np.array_equal(np.asarray(b["y"]).astype(int), np.asarray(y_te).astype(int))
    assert list(b["index"]) == list(D["Xte"].index), "BN prediction order differs from test design: STOP"
    p_bn = np.asarray(b["p"], float)
    add_cols = [c for c in D["Xtr"].columns if c.startswith("Age (bin)") or c.startswith("Debt Ratio (bin)")]
    ia = [list(D["Xtr"].columns).index(c) for c in add_cols]
    add = LogisticRegression(C=lr.C, penalty="l2", solver="saga", class_weight="balanced", max_iter=1000, random_state=42).fit(Xtr[:, ia], y_tr)
    p_add = add.predict_proba(Xte[:, ia])[:, 1]
    print("\nadditive diagnostic LR: %d one-hot columns (Age + Debt Ratio), C=%s l2, n_iter_ %s" % (len(ia), lr.C, add.n_iter_))

    # --- thresholds on OUT-OF-FOLD train predictions (LR, RF) ---
    t0 = time.time()
    oof = {"LR": cross_val_predict(clone(lr), Xtr, y_tr, cv=cv, method="predict_proba", n_jobs=1)[:, 1],
           "RF": cross_val_predict(clone(rf), Xtr, y_tr, cv=cv, method="predict_proba", n_jobs=1)[:, 1]}
    print("OOF predictions in %.0fs" % (time.time() - t0))
    grid = np.arange(0.01, 1.00, 0.01)
    def cost_at(p, y, t): 
        pr = p >= t; return 5 * int((~pr & (y == 1)).sum()) + int((pr & (y == 0)).sum())
    TAU = {"BN": 1 / 6}; oof_note = {}
    for k in ("LR", "RF"):
        cs = np.array([cost_at(oof[k], y_tr, t) for t in grid]); best = grid[cs == cs.min()]
        TAU[k] = float(best[0]); oof_note[k] = (float(best.min()), float(best.max()), int(cs.min()), int(5 * y_tr.sum()), int((y_tr == 0).sum()))
        print("%s OOF tau (5:1, 0.01 grid) = %.2f (plateau %.2f..%.2f) | OOF cost %d | OOF accept-all %d, reject-all %d" % (
            k, TAU[k], oof_note[k][0], oof_note[k][1], oof_note[k][2], oof_note[k][3], oof_note[k][4]))

    # --- test metrics ---
    y = np.asarray(y_te).astype(int); n = len(y)
    P = {"BN": p_bn, "LR": p_lr, "RF": p_rf, "LR_additive_AgeDebt (diagnostic)": p_add}
    names = list(P)
    rng = np.random.RandomState(42); B = 2000
    boot = {k: np.empty(B) for k in names}; t0 = time.time()
    for i in range(B):
        ix = rng.randint(0, n, n); yy = y[ix]
        for k in names: boot[k][i] = roc_auc_score(yy, P[k][ix])
    print("bootstrap (%d resamples, seed 42, shared) in %.0fs" % (B, time.time() - t0))
    rows = []
    for k in names:
        p = P[k]; a = roc_auc_score(y, p); lo, hi = np.percentile(boot[k], [2.5, 97.5])
        r = dict(model=k, auc=a, auc_ci_lo=lo, auc_ci_hi=hi, gini=2 * a - 1, ks=float(ks_2samp(p[y == 1], p[y == 0]).statistic),
                 ap=average_precision_score(y, p), mean_pred=float(p.mean()), obs_rate=float(y.mean()), distinct_probs=int(len(np.unique(p))))
        if k in TAU:
            t = TAU[k]; pr = p >= t
            TP, FP, FN, TN = int((pr & (y == 1)).sum()), int((pr & (y == 0)).sum()), int((~pr & (y == 1)).sum()), int((~pr & (y == 0)).sum())
            r.update(tau=t, tau_source="1/6 fixed" if k == "BN" else "OOF train", TP=TP, FP=FP, FN=FN, TN=TN, cost=5 * FN + FP,
                     n_flagged=int(pr.sum()), share_flagged=float(pr.mean()), accept_all_cost=int(5 * y.sum()), reject_all_cost=int((y == 0).sum()))
        rows.append(r)
    tbl = pd.DataFrame(rows)
    pd.set_option("display.width", 250, "display.max_columns", 40)
    print("\nTEST (n=%d, defaults %d)" % (n, y.sum()))
    print(tbl[["model", "auc", "auc_ci_lo", "auc_ci_hi", "gini", "ks", "ap", "mean_pred", "distinct_probs"]].to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("\nDecision results (pre-fixed thresholds; balanced-weight LR/RF probabilities are not calibrated, see mean_pred vs obs_rate %.4f):" % y.mean())
    print(tbl[tbl.tau.notna()][["model", "tau", "tau_source", "TP", "FP", "FN", "TN", "cost", "n_flagged", "share_flagged", "accept_all_cost", "reject_all_cost"]].to_string(index=False, float_format=lambda x: "%.4f" % x))

    comps = [("BN - LR", "BN", "LR"), ("BN - RF", "BN", "RF"), ("LR - RF", "LR", "RF"),
             ("BN - LR_additive (diagnostic)", "BN", "LR_additive_AgeDebt (diagnostic)")]
    brow = []
    for lab, a, c in comps:
        d = boot[a] - boot[c]; obs = roc_auc_score(y, P[a]) - roc_auc_score(y, P[c]); lo, hi = np.percentile(d, [2.5, 97.5])
        brow.append(dict(comparison=lab, diff=obs, ci_lo=lo, ci_hi=hi, p_value=max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001), n_resamples=B))
    bt = pd.DataFrame(brow)
    print("\nPaired bootstrap of AUC differences (same 2,000 resamples, seed 42; p floor 0.0005):")
    print(bt.to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("\ngrid-edge: RF chosen", rf_gs.best_params_, "-> n_estimators max edge, min_samples_leaf max edge, max_depth interior")

    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    s1 = wr(O_T, tbl.to_csv(index=False).encode()); s2 = wr(O_B, bt.to_csv(index=False).encode())
    s3 = wr(O_P, pickle.dumps(dict(index=list(D["Xte"].index), y=y, p_bn=p_bn, p_lr=p_lr, p_rf=p_rf, p_add=p_add,
                                    oof_lr=oof["LR"], oof_rf=oof["RF"], tau=TAU, lr_params=lr_gs.best_params_, rf_params=rf_gs.best_params_)))
    print("\nsaved sel_baselines_table.csv (%s), sel_baselines_bootstrap.csv (%s), sel_baseline_test_predictions.pkl (%s)" % (s1, s2, s3))
_c137()

In [ ]:
# Cell 138 (04_baselines_rq2.ipynb): Part B, EM vs listwise deletion (Default CPD). Writes sel_em_vs_deletion.csv + _metrics.json (exclusive create). No restart.
def _c138():
    import os, sys, json, pickle, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    from pgmpy.models import DiscreteBayesianNetwork
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "get_design" in globals(), "get_design not defined: re-run the Cell 136 definitions first. STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    O_C, O_M = "selection/tables/sel_em_vs_deletion.csv", "selection/tables/sel_em_vs_deletion_metrics.json"
    for rel in (O_C, O_M): ss.check_path(os.path.join(R, rel))

    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    def conv(df):
        out = {}
        for c in df.columns:
            s = df[c]
            if set(s.dropna().unique()) <= set(sn[c]): out[c] = s
            else:
                m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
                assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
                out[c] = m
        return pd.DataFrame(out)
    cc = conv(pd.read_pickle(R + "/Data/processed/sel_train_cc.pkl"))
    te = conv(pd.read_pickle(R + "/Data/processed/sel_test.pkl"))
    bn = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    model = DiscreteBayesianNetwork([tuple(e) for e in bn["model_edges"]]); model.add_cpds(*bn["cpds"]); assert model.check_model()
    cpd = model.get_cpds("Default")
    A, G = "Age (bin)", "Debt Ratio (bin)"
    assert set(cpd.variables[1:]) == {A, G} and len(cc) == 96251

    rows = []
    for a in sn[A]:
        for g in sn[G]:
            m = (cc[A] == a) & (cc[G] == g); n = int(m.sum()); k = int((cc.loc[m, "Default"] == 1).sum())
            rows.append(dict(age_bin=a, debt_ratio_bin=g, n_cc=n, defaults_cc=k, p_deletion=(k / n if n else np.nan),
                             p_em=float(cpd.get_value(**{"Default": 1, A: a, G: g}))))
    t = pd.DataFrame(rows)
    assert t.n_cc.min() > 0, "empty complete-case cell: STOP (report to planning instance)"
    t["abs_diff"] = (t.p_em - t.p_deletion).abs()
    pd.set_option("display.width", 220)
    print("Scoring uses only the Default CPD: Age and Debt Ratio are always observed and are Default's only parents (no children), so P(Default=1 | all evidence) = CPD lookup for both fits.")
    print("\n25-cell Default CPD, P(Default=1):"); print(t.to_string(index=False, float_format=lambda x: "%.4f" % x))
    big = t[t.abs_diff > 0.005]
    print("\nmax |EM - deletion| = %.4f | mean |diff| = %.4f | cells with |diff| > 0.005: %d" % (t.abs_diff.max(), t.abs_diff.mean(), len(big)))
    for _, r in big.iterrows(): print("   %s x %s: EM %.4f vs deletion %.4f (n_cc %d)" % (r.age_bin, r.debt_ratio_bin, r.p_em, r.p_deletion, r.n_cc))

    key = list(zip(te[A], te[G])); y = te["Default"].to_numpy().astype(int); n = len(y)
    d_em = {(r.age_bin, r.debt_ratio_bin): r.p_em for r in t.itertuples()}
    d_del = {(r.age_bin, r.debt_ratio_bin): r.p_deletion for r in t.itertuples()}
    p_em = np.array([d_em[k] for k in key]); p_del = np.array([d_del[k] for k in key])
    sv = pickle.load(open(R + "/selection/predictions/sel_bn_test_predictions.pkl", "rb"))
    print("EM lookup vs saved Phase 4 VE predictions: max |diff| = %.2e" % np.max(np.abs(p_em - np.asarray(sv["p"], float))))
    assert np.max(np.abs(p_em - np.asarray(sv["p"], float))) < 1e-10

    P = {"EM": p_em, "Deletion": p_del}
    rng = np.random.RandomState(42); B = 2000; boot = {k: np.empty(B) for k in P}
    for i in range(B):
        ix = rng.randint(0, n, n); yy = y[ix]
        for k in P: boot[k][i] = roc_auc_score(yy, P[k][ix])
    met = {}
    for k, p in P.items():
        a = roc_auc_score(y, p); lo, hi = np.percentile(boot[k], [2.5, 97.5])
        met[k] = dict(auc=a, ci=[float(lo), float(hi)], gini=2 * a - 1, ks=float(ks_2samp(p[y == 1], p[y == 0]).statistic),
                      ap=float(average_precision_score(y, p)), mean_pred=float(p.mean()), distinct=int(len(np.unique(p))))
        print("%-9s AUC %.4f (%.4f to %.4f) | Gini %.4f | KS %.4f | AP %.4f | mean predicted %.4f (observed %.4f) | distinct %d" % (
            k, a, lo, hi, 2 * a - 1, met[k]["ks"], met[k]["ap"], p.mean(), y.mean(), met[k]["distinct"]))
    d = boot["EM"] - boot["Deletion"]; obs = roc_auc_score(y, p_em) - roc_auc_score(y, p_del)
    lo, hi = np.percentile(d, [2.5, 97.5]); pv = max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001)
    print("\nPaired bootstrap AUC (EM - deletion): %+.4f [%+.4f, %+.4f], p = %.4f (2,000 resamples, seed 42, p floor 0.0005)" % (obs, lo, hi, pv))
    boot_res = dict(diff=float(obs), ci=[float(lo), float(hi)], p=float(pv))

    print("\nCost = r*FN + FP at tau_r = 1/(1+r), test set (flag if p >= tau_r):")
    grid = []
    for r in (5, 10, 15, 20):
        tau = 1 / (1 + r)
        for k, p in P.items():
            pr = p >= tau; FN = int((~pr & (y == 1)).sum()); FP = int((pr & (y == 0)).sum())
            grid.append(dict(r=r, tau=tau, fit=k, flagged=int(pr.sum()), TP=int((pr & (y == 1)).sum()), FP=FP, FN=FN,
                             cost=r * FN + FP, accept_all=int(r * y.sum()), reject_all=int((y == 0).sum())))
    gdf = pd.DataFrame(grid); print(gdf.to_string(index=False, float_format=lambda x: "%.4f" % x))

    D, y_tr, y_te = get_design()
    cols = list(D["Xtr"].columns); ia = [i for i, c in enumerate(cols) if c.startswith("Age (bin)") or c.startswith("Debt Ratio (bin)")]
    lb = LogisticRegression(C=1, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(D["Xtr"].values[:, ia], y_tr)
    a_lb = roc_auc_score(y_te, lb.predict_proba(D["Xte"].values[:, ia])[:, 1])
    print("\nDIAGNOSTIC: additive Age + Debt Ratio LR refit with lbfgs (C=1, converged in %s iterations): test AUC %.4f (saga fit in Cell 137 hit max_iter: 0.6228)" % (lb.n_iter_, a_lb))

    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    s1 = wr(O_C, t.to_csv(index=False).encode())
    s2 = wr(O_M, json.dumps(dict(metrics=met, paired_bootstrap_em_minus_deletion=boot_res, cost_grid=grid,
                                  max_abs_diff=float(t.abs_diff.max()), mean_abs_diff=float(t.abs_diff.mean()),
                                  cells_over_0_005=int(len(big)), additive_lbfgs_auc=float(a_lb),
                                  note="scoring via Default CPD lookup; Age and Debt Ratio always observed"), indent=1).encode())
    print("\nsaved sel_em_vs_deletion.csv (%s), sel_em_vs_deletion_metrics.json (%s)" % (s1, s2))
_c138()

In [ ]:
# Cell 139 (04_baselines_rq2.ipynb): Part C. Selection RQ2, VoI A/B, cost grid, corrected additive diagnostic. No restart.
def _c139():
    import os, sys, json, time, pickle, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from pgmpy.models import DiscreteBayesianNetwork
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "get_design" in globals(), "get_design not defined: re-run the Cell 136 definitions first. STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    spec = importlib.util.spec_from_file_location("sensitivity_saved", R + "/shared/src/sensitivity.py")
    sens = importlib.util.module_from_spec(spec); spec.loader.exec_module(sens)
    OUT = {"rq2": "selection/tables/sel_rq2_sensitivity_table.csv", "pkl": "selection/predictions/sel_rq2_per_row.pkl",
           "va": "selection/tables/sel_voi_partA_table.csv", "vb": "selection/tables/sel_voi_partB_table.csv",
           "grid": "selection/tables/sel_cost_ratio_grid.csv", "add": "selection/tables/sel_additive_lbfgs_diagnostic.csv"}
    for rel in OUT.values(): ss.check_path(os.path.join(R, rel))

    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    def conv(df):
        out = {}
        for c in df.columns:
            s = df[c]
            if set(s.dropna().unique()) <= set(sn[c]): out[c] = s
            else:
                m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
                assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
                out[c] = m
        return pd.DataFrame(out)
    te = conv(pd.read_pickle(R + "/Data/processed/sel_test.pkl"))
    b = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn.add_cpds(*b["cpds"]); assert bn.check_model()
    A, G = "Age (bin)", "Debt Ratio (bin)"
    V6 = [A, G, "Monthly Income (bin)", "Number of Dependents (bin)", "Real Estate Loans (bin)", "Open Credit Lines (bin)"]
    ZERO4 = V6[2:]
    STATES = {v: bn.get_cpds(v).state_names[v] for v in V6}
    evid = {}
    for idx, row in te.iterrows():
        e = {c: row[c] for c in V6 if pd.notnull(row[c])}
        e["M_Income"] = int(row["M_Income"]); e["M_Dependents"] = int(row["M_Dependents"])
        evid[idx] = e
    rows = list(te.index); assert len(set(rows)) == len(rows) == 30000
    y = te["Default"].to_numpy().astype(int)
    n_pat = len({tuple(sorted(e.items())) for e in evid.values()})
    print("distinct evidence patterns on test (observed variables + indicators): %d" % n_pat)

    t0 = time.time(); sens.analyze_blanket(bn, rows[:200], evid, V6, STATES, 1 / 6, skip_unobserved=True, verbose=False)
    proj = (time.time() - t0) / 200 * len(rows) / 60
    print("timing: projected full run <= %.1f min" % proj)
    if proj > 30: print("STOP: projected > 30 min."); return None
    t0 = time.time()
    res = sens.analyze_blanket(bn, rows, evid, V6, STATES, 1 / 6, skip_unobserved=True, verbose=True)
    print("full run: %d records in %.0fs" % (len(res), time.time() - t0))
    skipped_rows = sorted(set(rows) - set(res.row_index.unique()))
    print("rows excluded for zero-probability evidence:", len(skipped_rows), skipped_rows[:5])
    print("records per variable:", res.groupby("variable").size().rename(lambda v: v.replace(" (bin)", "")).to_dict())

    # zero-sensitivity assertions
    print("\nmax over rows of S_exp / S_max / delta for the four non-blanket variables (asserted < 1e-12):")
    for v in ZERO4:
        g = res[res.variable == v]
        mx = [float(np.nanmax(g[c])) for c in ("S_exp", "S_max", "delta")]
        print("  %-26s S_exp %.3e | S_max %.3e | delta %.3e | n %d" % (v.replace(" (bin)", ""), *mx, len(g)))
        assert max(mx) < 1e-12, "non-blanket variable shows non-zero sensitivity: STOP"
    print("skipped states (Smax) for the six variables:", int(res.skipped.sum()))

    # RQ2 table
    rng_seed = 42; B = 2000; trows = []
    for v in V6:
        g = res[res.variable == v]; x = g.S_exp.dropna().to_numpy()
        rg = np.random.RandomState(rng_seed); bm = np.array([x[rg.randint(0, len(x), len(x))].mean() for _ in range(B)])
        lo, hi = np.percentile(bm, [2.5, 97.5])
        trows.append(dict(variable=v.replace(" (bin)", ""), in_blanket=v in (A, G), n_rows=len(x), mean_S_exp=x.mean(), ci_lo=lo, ci_hi=hi,
                          median=float(np.median(x)), p99=float(np.percentile(x, 99)), max=float(x.max()),
                          share_gt_0_01=float((x > 0.01).mean()), share_gt_0_05=float((x > 0.05).mean()), n_patterns=n_pat))
    rq2 = pd.DataFrame(trows).sort_values("mean_S_exp", ascending=False)
    pd.set_option("display.width", 220, "display.max_columns", 30)
    print("\n=== Selection RQ2: S_exp per variable (test, bootstrap 2,000, seed 42) ===")
    print(rq2.to_string(index=False, float_format=lambda x: "%.5f" % x))

    # VoI Part A, Age and Debt Ratio, cost-ratio grid
    ok_rows = np.array([i not in set(skipped_rows) for i in rows])
    ga = res[res.variable == A].set_index("row_index"); gg = res[res.variable == G].set_index("row_index")
    keep = [i for i in rows if i not in set(skipped_rows)]
    assert list(ga.index) == keep and list(gg.index) == keep
    base = ga.baseline_prob.to_numpy()
    delta = np.column_stack([ga.delta.to_numpy(), gg.delta.to_numpy()])
    ret = np.column_stack([ga.retracted_prob.to_numpy(), gg.retracted_prob.to_numpy()])
    assert np.isfinite(ret).all(), "NaN retracted posterior: report"
    yk = y[ok_rows]; nk = len(yk)
    taus = {r: 1 / (1 + r) for r in (5, 10, 15, 20)}
    Fm = {r: np.column_stack([((base >= t) != (ret[:, j] >= t)).astype(float) for j in range(2)]) for r, t in taus.items()}
    mats = [delta] + [Fm[r] for r in taus]
    bm = sens.boot_means(mats, B=2000, seed=42)
    dlo, dhi = np.percentile(bm[0], [2.5, 97.5], axis=0)
    vrows = []
    for j, v in enumerate((A, G)):
        for k, (r, t) in enumerate(taus.items()):
            flo, fhi = np.percentile(bm[1 + k][:, j], [2.5, 97.5])
            vrows.append(dict(variable=v.replace(" (bin)", ""), r=r, tau=t, n_rows=nk, mean_delta=delta[:, j].mean(), delta_ci_lo=dlo[j], delta_ci_hi=dhi[j],
                              median_delta=float(np.median(delta[:, j])), n_flips=int(Fm[r][:, j].sum()), flip_rate=Fm[r][:, j].mean(), flip_ci_lo=flo, flip_ci_hi=fhi,
                              share_baseline_ge_tau=float((base >= t).mean())))
    va = pd.DataFrame(vrows)
    print("\n=== VoI Part A: retract Age / Debt Ratio; flips at tau_r = 1/(1+r), grid fixed a priori ===")
    print(va.to_string(index=False, float_format=lambda x: "%.5f" % x))
    chk = res[res.variable.isin([A, G])]
    assert int(((chk.baseline_prob >= 1 / 6) != (chk.retracted_prob >= 1 / 6)).sum()) == int(chk.flip.sum())

    # BN cost grid (test)
    pbn = pickle.load(open(R + "/selection/predictions/sel_bn_test_predictions.pkl", "rb"))
    p = np.asarray(pbn["p"], float); assert np.array_equal(np.asarray(pbn["y"]).astype(int), y)
    grows = []
    for r, t in taus.items():
        pr = p >= t; FN = int((~pr & (y == 1)).sum()); FP = int((pr & (y == 0)).sum()); c = r * FN + FP
        aa, ra = int(r * y.sum()), int((y == 0).sum())
        grows.append(dict(r=r, tau=t, flagged=int(pr.sum()), TP=int((pr & (y == 1)).sum()), FP=FP, FN=FN, cost=c, accept_all=aa, reject_all=ra,
                          saving_vs_accept_all=1 - c / aa, saving_vs_reject_all=1 - c / ra))
    grid = pd.DataFrame(grows)
    print("\n=== BN cost on test at tau_r (same as Cell 138 EM rows: 10025 / 18069 / 21788 / 23602) ===")
    print(grid.to_string(index=False, float_format=lambda x: "%.4f" % x))
    assert list(grid.cost) == [10025, 18069, 21788, 23602], "cost grid differs from Cell 138: STOP"

    # VoI Part B
    post = sens.make_posterior_fn(bn); brow = []
    print("\n=== VoI Part B: naturally missing Income / Dependents (observe the value; EPS = EVOI = 0?) ===")
    for m_var, val_var in (("M_Income", "Monthly Income (bin)"), ("M_Dependents", "Number of Dependents (bin)")):
        n_, worst, flips, sk = sens.partB_check(post, keep, evid, m_var, val_var, STATES, 1 / 6)
        print("  %-26s %d incomplete rows | max shift %.3e | flips at 1/6 %d | skipped states %d" % (val_var.replace(" (bin)", ""), n_, worst, flips, sk))
        assert worst < 1e-12 and flips == 0, "EPS/EVOI not zero: STOP"
        brow.append(dict(variable=val_var.replace(" (bin)", ""), n_incomplete_rows=n_, max_shift=worst, flips_at_1_6=flips, skipped_states=sk,
                         EPS_zero=worst < 1e-12, EVOI_zero=flips == 0))
    vb = pd.DataFrame(brow)

    # corrected additive diagnostic (lbfgs)
    D, y_tr, y_te = get_design(); cols = list(D["Xtr"].columns)
    ia = [i for i, c in enumerate(cols) if c.startswith("Age (bin)") or c.startswith("Debt Ratio (bin)")]
    lb = LogisticRegression(C=1, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(D["Xtr"].values[:, ia], y_tr)
    pa = lb.predict_proba(D["Xte"].values[:, ia])[:, 1]; assert np.array_equal(np.asarray(y_te).astype(int), y)
    rg = np.random.RandomState(42); d = np.empty(2000)
    for i in range(2000):
        ix = rg.randint(0, len(y), len(y)); yy = y[ix]; d[i] = roc_auc_score(yy, p[ix]) - roc_auc_score(yy, pa[ix])
    obs = roc_auc_score(y, p) - roc_auc_score(y, pa); lo, hi = np.percentile(d, [2.5, 97.5])
    add = pd.DataFrame([dict(model="LR additive Age+Debt Ratio, lbfgs C=1 (converged)", auc=roc_auc_score(y, pa), bn_minus_additive=obs, ci_lo=lo, ci_hi=hi,
                             p_value=max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001), n_iter=int(lb.n_iter_[0]),
                             note="replaces the saga row in sel_baselines_table/bootstrap, which hit max_iter")])
    print("\nCORRECTED additive diagnostic: AUC %.4f | BN - additive %+.4f [%+.4f, %+.4f], p = %.4f" % (add.auc[0], obs, lo, hi, add.p_value[0]))

    def wr(rel, data):
        pth = ss.check_path(os.path.join(R, rel))
        with open(pth, "xb") as f: f.write(data)
        assert hashlib.sha256(open(pth, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    sh = {k: wr(OUT[k], v) for k, v in [("rq2", rq2.to_csv(index=False).encode()), ("va", va.to_csv(index=False).encode()), ("vb", vb.to_csv(index=False).encode()),
                                         ("grid", grid.to_csv(index=False).encode()), ("add", add.to_csv(index=False).encode()),
                                         ("pkl", pickle.dumps(dict(per_row=res, n_patterns=n_pat, excluded_rows=skipped_rows, tau_grid=taus)))]}
    print("\nsaved:", {OUT[k].split("/")[-1]: v for k, v in sh.items()})
    return True
_c139()

In [ ]:
# Cell 140 (04_baselines_rq2.ipynb): diagnose NaN retracted/S_exp rows in Part C. Read-only, writes NOTHING. No restart.
def _c140():
    import os, sys, json, pickle, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from pgmpy.models import DiscreteBayesianNetwork
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    spec = importlib.util.spec_from_file_location("sensitivity_saved", R + "/shared/src/sensitivity.py")
    sens = importlib.util.module_from_spec(spec); spec.loader.exec_module(sens)
    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    raw = pd.read_pickle(R + "/Data/processed/sel_test.pkl")
    te = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c]) else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    b = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn.add_cpds(*b["cpds"])
    A, G = "Age (bin)", "Debt Ratio (bin)"
    V6 = [A, G, "Monthly Income (bin)", "Number of Dependents (bin)", "Real Estate Loans (bin)", "Open Credit Lines (bin)"]
    STATES = {v: bn.get_cpds(v).state_names[v] for v in V6}
    evid = {}
    for idx, row in te.iterrows():
        e = {c: row[c] for c in V6 if pd.notnull(row[c])}; e["M_Income"] = int(row["M_Income"]); e["M_Dependents"] = int(row["M_Dependents"]); evid[idx] = e
    res = sens.analyze_blanket(bn, list(te.index), evid, V6, STATES, 1 / 6, skip_unobserved=True, verbose=False)
    bad = res[res.S_exp.isna() | res.retracted_prob.isna()]
    print("records with NaN S_exp or NaN retracted_prob: %d" % len(bad))
    print(bad.assign(variable=bad.variable.str.replace(" (bin)", ""))[["row_index", "variable", "S_exp", "S_max", "retracted_prob", "baseline_prob", "skipped"]].to_string(index=False))
    print("\nby variable:", bad.groupby("variable").size().rename(lambda v: v.replace(" (bin)", "")).to_dict())
    print("NaN retracted_prob only:", res[res.retracted_prob.isna()].groupby("variable").size().rename(lambda v: v.replace(" (bin)", "")).to_dict())
    nodes = list(bn.nodes())
    for _, r in bad.drop_duplicates(["row_index", "variable"]).iterrows():
        e = {k: v for k, v in evid[r.row_index].items() if k != r.variable}
        zeros = []
        for n in nodes:
            cpd = bn.get_cpds(n); vs = list(cpd.variables)
            if n == "Default" or any(v not in e for v in vs): continue
            idx = tuple(pd.Index(cpd.state_names[v]).get_indexer([e[v]])[0] for v in vs)
            if float(np.asarray(cpd.values)[idx]) == 0: zeros.append((n.replace(" (bin)", ""), [v.replace(" (bin)", "") for v in vs[1:]]))
        print("row %s, retract %s: exact-zero CPT entries fully inside the remaining evidence: %s | retracted p = %s" % (
            r.row_index, r.variable.replace(" (bin)", ""), zeros or "none (zero arises through marginalisation)", r.retracted_prob))
_c140()

In [ ]:
# Cell 141 (04_baselines_rq2.ipynb): exact (tensor) check of the 5 NaN records vs pgmpy VE. Read-only, writes NOTHING. No restart.
def _c141():
    import os, json, pickle, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    raw = pd.read_pickle(R + "/Data/processed/sel_test.pkl")
    te = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c]) else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    b = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn.add_cpds(*b["cpds"])
    nodes = list(bn.nodes()); S = {n: list(bn.get_cpds(n).state_names[n]) for n in nodes}
    J = np.ones([len(S[n]) for n in nodes])
    for n in nodes:
        cpd = bn.get_cpds(n); vs = list(cpd.variables); arr = np.asarray(cpd.values, float)
        for v in vs: assert list(cpd.state_names[v]) == S[v], "state order mismatch %s" % v
        order = sorted(range(len(vs)), key=lambda i: nodes.index(vs[i]))
        J = J * arr.transpose(order).reshape([len(S[m]) if m in vs else 1 for m in nodes])
    print("joint tensor sum = %.12f | cells %d | min positive %.3e" % (J.sum(), J.size, J[J > 0].min()))
    def sl(ev, extra=None):
        d = dict(ev); d.update(extra or {})
        return tuple(S[n].index(d[n]) if n in d else slice(None) for n in nodes)
    eng = VariableElimination(bn)
    V6 = ["Age (bin)", "Debt Ratio (bin)", "Monthly Income (bin)", "Number of Dependents (bin)", "Real Estate Loans (bin)", "Open Credit Lines (bin)"]
    for row, var in [(56242, "Age (bin)"), (70067, "Age (bin)"), (56242, "Number of Dependents (bin)"), (122714, "Real Estate Loans (bin)")]:
        r = te.loc[row]
        ev = {c: r[c] for c in V6 if pd.notnull(r[c])}; ev["M_Income"] = int(r["M_Income"]); ev["M_Dependents"] = int(r["M_Dependents"])
        er = {k: v for k, v in ev.items() if k != var}
        Pe, Pr = J[sl(ev)].sum(), J[sl(er)].sum()
        post_full = J[sl(ev, {"Default": 1})].sum() / Pe; post_ret = J[sl(er, {"Default": 1})].sum() / Pr
        w = {s: J[sl(er, {var: s})].sum() / Pr for s in S[var]}
        print("\nrow %s retract %s: P(full evidence)=%.3e | P(retracted evidence)=%.3e | exact posterior full %.6f retracted %.6f" % (row, var.replace(" (bin)", ""), Pe, Pr, post_full, post_ret))
        print("  exact predictive weights:", {k: round(v, 6) for k, v in w.items()}, "| sum %.6f" % sum(w.values()))
        try:
            p = eng.query(variables=["Default"], evidence=er, show_progress=False)
            print("  pgmpy VE Default posterior given retracted evidence:", np.asarray(p.values))
        except Exception as e:
            print("  pgmpy VE raised:", type(e).__name__, str(e)[:160])
        try:
            f = eng.query(variables=[var], evidence=er, show_progress=False)
            print("  pgmpy VE predictive weights:", np.asarray(f.values))
        except Exception as e:
            print("  pgmpy VE predictive raised:", type(e).__name__, str(e)[:160])
_c141()

In [ ]:
# Cell 142 (04_baselines_rq2.ipynb): resolve tensor-vs-VE contradiction on the 3 rows. Read-only, writes NOTHING. No restart.
def _c142():
    import os, json, pickle, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    spec = importlib.util.spec_from_file_location("ll", R + "/shared/src/loglik.py"); ll = importlib.util.module_from_spec(spec); spec.loader.exec_module(ll)
    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    raw = pd.read_pickle(R + "/Data/processed/sel_test.pkl")
    te = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c]) else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    b = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn.add_cpds(*b["cpds"])
    nodes = list(bn.nodes()); S = {n: list(bn.get_cpds(n).state_names[n]) for n in nodes}
    print("node order:", [n.replace(" (bin)", "") for n in nodes], "| state counts", [len(S[n]) for n in nodes])
    J = np.ones([len(S[n]) for n in nodes])
    for n in nodes:
        cpd = bn.get_cpds(n); vs = list(cpd.variables); arr = np.asarray(cpd.values, float)
        order = sorted(range(len(vs)), key=lambda i: nodes.index(vs[i]))
        J = J * arr.transpose(order).reshape([len(S[m]) if m in vs else 1 for m in nodes])
    print("joint shape", J.shape, "sum %.12f" % J.sum())
    eng = VariableElimination(bn)
    def evid(i):
        r = te.loc[i]; return {c: r[c] for c in nodes if c != "Default" and pd.notnull(r[c])}
    def tens(ev): return J[tuple(S[n].index(ev[n]) if n in ev else slice(None) for n in nodes)].sum()
    rng = np.random.RandomState(42); ok = rng.choice(te.index.values, 20, replace=False); worst = 0
    for i in ok:
        ev = evid(i); Pe = tens(ev)
        pt = J[tuple(S[n].index(ev[n]) if n in ev else (1 if n == "Default" else slice(None)) for n in nodes)].sum() / Pe
        pv = eng.query(variables=["Default"], evidence=ev, show_progress=False).get_value(Default=1)
        worst = max(worst, abs(pt - pv))
    print("tensor vs VE posterior on 20 ordinary rows: max |diff| = %.3e  (tensor valid if ~1e-15)" % worst)
    for i in (56242, 70067, 122714):
        ev = evid(i)
        print("\nrow %s evidence: %s" % (i, {k.replace(" (bin)", ""): v for k, v in ev.items()}))
        print("  missing nodes (not in evidence, excl Default):", [n.replace(" (bin)", "") for n in nodes if n not in ev and n != "Default"])
        print("  tensor P(evidence) = %.6e" % tens(ev))
        obs = dict(ev)
        try: print("  VE joint-marginal P(evidence) = %.6e" % eng.query(variables=list(obs), evidence=None, show_progress=False).get_value(**obs))
        except Exception as e: print("  VE joint-marginal raised", type(e).__name__)
        row = pd.DataFrame([{**{n: np.nan for n in nodes}, **ev, "Default": np.nan}])
        print("  loglik.py log P(evidence) (Default missing too) = %.4f -> P = %.6e" % (ll.compute_log_likelihood(bn, row[nodes]), np.exp(ll.compute_log_likelihood(bn, row[nodes]))))
        for n in nodes:
            if n == "Default": continue
            cpd = bn.get_cpds(n); vs = list(cpd.variables)
            if all(v in ev for v in vs):
                print("   %-24s CPT value %.3e" % (n.replace(" (bin)", ""), float(np.asarray(cpd.values)[tuple(S[v].index(ev[v]) for v in vs)])))
_c142()

In [ ]:
# Cell 143 (04_baselines_rq2.ipynb): Part C v2 (NaN-aware). Replaces Cell 139. Writes 6 files (exclusive create). No restart.
def _c143():
    import os, sys, json, time, pickle, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from pgmpy.models import DiscreteBayesianNetwork
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "get_design" in globals(), "get_design not defined: re-run the Cell 136 definitions first. STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    spec = importlib.util.spec_from_file_location("sensitivity_saved", R + "/shared/src/sensitivity.py")
    sens = importlib.util.module_from_spec(spec); spec.loader.exec_module(sens)
    OUT = {"rq2": "selection/tables/sel_rq2_sensitivity_table.csv", "pkl": "selection/predictions/sel_rq2_per_row.pkl",
           "va": "selection/tables/sel_voi_partA_table.csv", "vb": "selection/tables/sel_voi_partB_table.csv",
           "grid": "selection/tables/sel_cost_ratio_grid.csv", "add": "selection/tables/sel_additive_lbfgs_diagnostic.csv"}
    for rel in OUT.values(): ss.check_path(os.path.join(R, rel))

    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    raw = pd.read_pickle(R + "/Data/processed/sel_test.pkl")
    te = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c]) else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    b = pickle.load(open(R + "/selection/models/sel_bn_final.pkl", "rb"))
    bn = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn.add_cpds(*b["cpds"]); assert bn.check_model()
    A, G = "Age (bin)", "Debt Ratio (bin)"
    V6 = [A, G, "Monthly Income (bin)", "Number of Dependents (bin)", "Real Estate Loans (bin)", "Open Credit Lines (bin)"]
    ZERO4 = V6[2:]
    STATES = {v: bn.get_cpds(v).state_names[v] for v in V6}
    evid = {}
    for idx, row in te.iterrows():
        e = {c: row[c] for c in V6 if pd.notnull(row[c])}
        e["M_Income"] = int(row["M_Income"]); e["M_Dependents"] = int(row["M_Dependents"]); evid[idx] = e
    rows = list(te.index); assert len(set(rows)) == len(rows) == 30000
    y = te["Default"].to_numpy().astype(int)
    n_pat = len({tuple(sorted(e.items())) for e in evid.values()})
    print("distinct evidence patterns on test: %d" % n_pat)

    t0 = time.time()
    res = sens.analyze_blanket(bn, rows, evid, V6, STATES, 1 / 6, skip_unobserved=True, verbose=False)
    print("full run: %d records in %.0fs" % (len(res), time.time() - t0))
    skipped_rows = sorted(set(rows) - set(res.row_index.unique()))
    assert skipped_rows == [], "unexpected NaN baseline rows: STOP"
    nanrec = res[res.S_exp.isna() | res.retracted_prob.isna()]
    print("records with NaN S_exp or NaN retracted posterior: %d (rows %s)" % (len(nanrec), sorted(nanrec.row_index.unique())))
    print(nanrec.assign(variable=nanrec.variable.str.replace(" (bin)", ""))[["row_index", "variable", "S_exp", "retracted_prob"]].to_string(index=False))

    print("\nmax over rows of S_exp / S_max / delta, four non-blanket variables (asserted < 1e-12):")
    for v in ZERO4:
        g = res[res.variable == v]
        mx = [float(np.nanmax(g[c])) for c in ("S_exp", "S_max", "delta")]
        print("  %-26s S_exp %.3e | S_max %.3e | delta %.3e | n %d" % (v.replace(" (bin)", ""), *mx, len(g)))
        assert max(mx) < 1e-12, "non-blanket variable shows non-zero sensitivity: STOP"

    B = 2000; trows = []
    for v in V6:
        g = res[res.variable == v]; x = g.S_exp.dropna().to_numpy()
        rg = np.random.RandomState(42); bm = np.array([x[rg.randint(0, len(x), len(x))].mean() for _ in range(B)])
        lo, hi = np.percentile(bm, [2.5, 97.5])
        trows.append(dict(variable=v.replace(" (bin)", ""), in_blanket=v in (A, G), n_records=len(g), n_rows=len(x), n_nan_S_exp=int(g.S_exp.isna().sum()),
                          mean_S_exp=x.mean(), ci_lo=lo, ci_hi=hi, median=float(np.median(x)), p99=float(np.percentile(x, 99)), max=float(x.max()),
                          share_gt_0_01=float((x > 0.01).mean()), share_gt_0_05=float((x > 0.05).mean()), n_patterns=n_pat))
    rq2 = pd.DataFrame(trows).sort_values("mean_S_exp", ascending=False)
    pd.set_option("display.width", 250, "display.max_columns", 30)
    print("\n=== Selection RQ2: S_exp per variable (test, NaN-aware, bootstrap 2,000, seed 42) ===")
    print(rq2.to_string(index=False, float_format=lambda x: "%.5f" % x))

    ga = res[res.variable == A].set_index("row_index"); gg = res[res.variable == G].set_index("row_index")
    assert list(ga.index) == rows and list(gg.index) == rows
    base = ga.baseline_prob.to_numpy()
    delta = np.column_stack([ga.delta.to_numpy(), gg.delta.to_numpy()])
    ret = np.column_stack([ga.retracted_prob.to_numpy(), gg.retracted_prob.to_numpy()])
    fin = np.isfinite(ret).all(axis=1); nk = int(fin.sum())
    print("\nVoI Part A uses the %d rows where both retracted posteriors are finite (%d rows dropped: %s)" % (nk, 30000 - nk, [r for r, f in zip(rows, fin) if not f]))
    base, delta, ret = base[fin], delta[fin], ret[fin]
    taus = {r: 1 / (1 + r) for r in (5, 10, 15, 20)}
    Fm = {r: np.column_stack([((base >= t) != (ret[:, j] >= t)).astype(float) for j in range(2)]) for r, t in taus.items()}
    bm = sens.boot_means([delta] + [Fm[r] for r in taus], B=2000, seed=42)
    dlo, dhi = np.percentile(bm[0], [2.5, 97.5], axis=0)
    vrows = []
    for j, v in enumerate((A, G)):
        for k, (r, t) in enumerate(taus.items()):
            flo, fhi = np.percentile(bm[1 + k][:, j], [2.5, 97.5])
            vrows.append(dict(variable=v.replace(" (bin)", ""), r=r, tau=t, n_rows=nk, mean_delta=delta[:, j].mean(), delta_ci_lo=dlo[j], delta_ci_hi=dhi[j],
                              median_delta=float(np.median(delta[:, j])), n_flips=int(Fm[r][:, j].sum()), flip_rate=Fm[r][:, j].mean(), flip_ci_lo=flo, flip_ci_hi=fhi,
                              share_baseline_ge_tau=float((base >= t).mean())))
    va = pd.DataFrame(vrows)
    print("\n=== VoI Part A: retract Age / Debt Ratio; flips at tau_r = 1/(1+r) (grid fixed a priori) ===")
    print(va.to_string(index=False, float_format=lambda x: "%.5f" % x))
    chk = res[res.variable.isin([A, G]) & res.retracted_prob.notna()]
    assert int(((chk.baseline_prob >= 1 / 6) != (chk.retracted_prob >= 1 / 6)).sum()) == int(chk.flip.sum())

    pbn = pickle.load(open(R + "/selection/predictions/sel_bn_test_predictions.pkl", "rb"))
    p = np.asarray(pbn["p"], float); assert np.array_equal(np.asarray(pbn["y"]).astype(int), y)
    grows = []
    for r, t in taus.items():
        pr = p >= t; FN = int((~pr & (y == 1)).sum()); FP = int((pr & (y == 0)).sum()); c = r * FN + FP
        aa, ra = int(r * y.sum()), int((y == 0).sum())
        grows.append(dict(r=r, tau=t, flagged=int(pr.sum()), TP=int((pr & (y == 1)).sum()), FP=FP, FN=FN, cost=c, accept_all=aa, reject_all=ra,
                          saving_vs_accept_all=1 - c / aa, saving_vs_reject_all=1 - c / ra))
    grid = pd.DataFrame(grows)
    print("\n=== BN cost on test at tau_r (all 30,000 rows) ===")
    print(grid.to_string(index=False, float_format=lambda x: "%.4f" % x))
    assert list(grid.cost) == [10025, 18069, 21788, 23602], "cost grid differs from Cell 138: STOP"

    post = sens.make_posterior_fn(bn); brow = []
    print("\n=== VoI Part B: naturally missing Income / Dependents (observe the value) ===")
    for m_var, val_var in (("M_Income", "Monthly Income (bin)"), ("M_Dependents", "Number of Dependents (bin)")):
        n_, worst, flips, sk = sens.partB_check(post, rows, evid, m_var, val_var, STATES, 1 / 6)
        print("  %-22s %d incomplete rows | max shift %.3e | flips at 1/6 %d | skipped states (NaN posterior) %d" % (val_var.replace(" (bin)", ""), n_, worst, flips, sk))
        assert worst < 1e-12 and flips == 0, "EPS/EVOI not zero: STOP"
        brow.append(dict(variable=val_var.replace(" (bin)", ""), n_incomplete_rows=n_, max_shift=worst, flips_at_1_6=flips, skipped_states=sk,
                         EPS_zero=worst < 1e-12, EVOI_zero=flips == 0))
    vb = pd.DataFrame(brow)

    D, y_tr, y_te = get_design(); cols = list(D["Xtr"].columns)
    ia = [i for i, c in enumerate(cols) if c.startswith("Age (bin)") or c.startswith("Debt Ratio (bin)")]
    lb = LogisticRegression(C=1, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(D["Xtr"].values[:, ia], y_tr)
    pa = lb.predict_proba(D["Xte"].values[:, ia])[:, 1]; assert np.array_equal(np.asarray(y_te).astype(int), y)
    rg = np.random.RandomState(42); d = np.empty(2000)
    for i in range(2000):
        ix = rg.randint(0, len(y), len(y)); yy = y[ix]; d[i] = roc_auc_score(yy, p[ix]) - roc_auc_score(yy, pa[ix])
    obs = roc_auc_score(y, p) - roc_auc_score(y, pa); lo, hi = np.percentile(d, [2.5, 97.5])
    add = pd.DataFrame([dict(model="LR additive Age+Debt Ratio, lbfgs C=1 (converged)", auc=roc_auc_score(y, pa), bn_minus_additive=obs, ci_lo=lo, ci_hi=hi,
                             p_value=max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001), n_iter=int(lb.n_iter_[0]),
                             note="replaces the saga row in sel_baselines_table/bootstrap, which hit max_iter")])
    print("\nCORRECTED additive diagnostic: AUC %.4f | BN - additive %+.4f [%+.4f, %+.4f], p = %.4f" % (add.auc[0], obs, lo, hi, add.p_value[0]))

    def wr(rel, data):
        pth = ss.check_path(os.path.join(R, rel))
        with open(pth, "xb") as f: f.write(data)
        assert hashlib.sha256(open(pth, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    sh = {k: wr(OUT[k], v) for k, v in [("rq2", rq2.to_csv(index=False).encode()), ("va", va.to_csv(index=False).encode()), ("vb", vb.to_csv(index=False).encode()),
                                         ("grid", grid.to_csv(index=False).encode()), ("add", add.to_csv(index=False).encode()),
                                         ("pkl", pickle.dumps(dict(per_row=res, n_patterns=n_pat, nan_records=nanrec, tau_grid=taus)))]}
    print("\nsaved:", {OUT[k].split("/")[-1]: v for k, v in sh.items()})
_c143()

In [ ]:
# Cell 144 (04_baselines_rq2.ipynb): all Phase 5 figures from saved results. Exclusive create. No restart.
def _c144():
    import os, sys, io, pickle, hashlib, textwrap, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, roc_auc_score
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"
    NAMES = ["sel_default_heatmap.png", "sel_calibration.png", "sel_cost_vs_ratio.png", "sel_flip_rates_by_ratio.png",
             "sel_rq2_sensitivity_distributions.png", "sel_roc_curves.png", "sel_rq2_sensitivity_ranking.png"]
    for n in NAMES: ss.check_path(os.path.join(R, F + n))
    T = R + "/selection/tables/"
    cells = pd.read_csv(T + "sel_default_cells.csv"); grid = pd.read_csv(T + "sel_cost_ratio_grid.csv")
    va = pd.read_csv(T + "sel_voi_partA_table.csv"); rq2 = pd.read_csv(T + "sel_rq2_sensitivity_table.csv")
    bp = pickle.load(open(R + "/selection/predictions/sel_baseline_test_predictions.pkl", "rb"))
    per = pickle.load(open(R + "/selection/predictions/sel_rq2_per_row.pkl", "rb"))["per_row"]
    y = np.asarray(bp["y"]).astype(int); P = {"LR": np.asarray(bp["p_lr"]), "RF": np.asarray(bp["p_rf"]), "BN": np.asarray(bp["p_bn"])}
    COL = {"LR": "tab:blue", "RF": "tab:orange", "BN": "tab:green"}
    PAL = {"Age": "#0072B2", "Debt Ratio": "#E69F00"}; REJ = "#D55E00"
    short = lambda s: s.replace(" (bin)", "")
    def finish(fig, ax_list, cap):
        for ax in ax_list:
            ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        fig.tight_layout(rect=[0, 0.09, 1, 1])
        fig.text(0.01, 0.005, textwrap.fill(cap, 140), fontsize=8, va="bottom")
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    out = {}

    # 1 heatmap
    ages = list(dict.fromkeys(cells.age_bin)); drs = list(dict.fromkeys(cells.debt_ratio_bin))
    M = cells.pivot(index="age_bin", columns="debt_ratio_bin", values="p_default_em").reindex(index=ages, columns=drs)
    N = cells.pivot(index="age_bin", columns="debt_ratio_bin", values="n").reindex(index=ages, columns=drs)
    mx = float(np.nanmax(M.values)); assert round(mx, 4) == 0.1194, "max cell differs from 0.1194: STOP"
    fig, ax = plt.subplots(figsize=(10, 6.5))
    im = ax.imshow(M.values, cmap="YlOrRd", vmin=0, vmax=0.18, aspect="auto")
    for i in range(5):
        for j in range(5):
            v = M.values[i, j]; ax.text(j, i, "%.4f\nn=%s" % (v, format(int(N.values[i, j]), ",")), ha="center", va="center", fontsize=9, color="white" if v > 0.10 else "black")
    ax.set_xticks(range(5)); ax.set_xticklabels(drs); ax.set_yticks(range(5)); ax.set_yticklabels(ages)
    ax.set_xlabel("Debt Ratio bin"); ax.set_ylabel("Age bin")
    cb = fig.colorbar(im, ax=ax, pad=0.02); cb.set_label("P(Default=1 | Age, Debt Ratio)")
    cb.ax.axhline(1 / 6, color="black", lw=2); cb.ax.text(1.15, 1 / 6, "tau = 1/6", transform=cb.ax.get_yaxis_transform(), va="center", fontsize=9)
    ax.set_title("Selection BN: P(Default=1 | Age bin, Debt Ratio bin), EM model, training counts n")
    out[NAMES[0]] = finish(fig, [], "Maximum cell probability is %.4f, below tau = 1/6 (5:1 FN:FP cost ratio, marked on the colour scale), so no cell is flagged. Cell n = rows of sel_train_full (119,999)." % mx)

    # 2 calibration
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.8))
    d = pd.DataFrame({"p": P["BN"], "y": y}).groupby("p").agg(obs=("y", "mean"), n=("y", "size")).reset_index()
    a1.plot([0, 0.15], [0, 0.15], color="gray", ls=":", label="Perfect calibration")
    a1.scatter(d.p, d.obs, s=d.n / 12, color=COL["BN"], alpha=0.8, label="BN, one point per distinct value (25), size ~ n")
    a1.set_xlim(0, 0.15); a1.set_ylim(0, 0.15); a1.set_xlabel("Predicted P(Default=1)"); a1.set_ylabel("Observed default rate"); a1.set_title("BN (25 distinct predicted values)"); a1.legend(loc="upper left", frameon=False, fontsize=9)
    a2.plot([0, 1], [0, 1], color="gray", ls=":", label="Perfect calibration")
    for k in ("LR", "RF"):
        q = pd.qcut(pd.Series(P[k]), 10, duplicates="drop"); g = pd.DataFrame({"p": P[k], "y": y}).groupby(q.values).mean()
        a2.plot(g.p, g.y, marker="o", color=COL[k], label=k + " (deciles of predicted probability)")
    a2.set_xlim(0, 1); a2.set_ylim(0, 1); a2.set_xlabel("Predicted P(Default=1)"); a2.set_ylabel("Observed default rate"); a2.set_title("LR and RF (class_weight balanced)"); a2.legend(loc="upper left", frameon=False, fontsize=9)
    out[NAMES[1]] = finish(fig, [a1, a2], "Test set, n = 30,000, observed default rate %.4f. Mean predicted: BN %.4f, LR %.4f, RF %.4f. Balanced class weights inflate LR and RF probabilities, so only their ranking is meaningful; the BN is calibrated." % (y.mean(), P["BN"].mean(), P["LR"].mean(), P["RF"].mean()))

    # 3 cost vs ratio
    fig, ax = plt.subplots(figsize=(10, 6.2))
    ax.plot(grid.r, grid.cost, marker="o", color=COL["BN"], label="BN at tau_r = 1/(1+r)")
    ax.plot(grid.r, grid.accept_all, marker="s", color=REJ, label="Accept-all (flag nobody)")
    ax.plot(grid.r, grid.reject_all, marker="^", color="dimgray", label="Reject-all (flag everybody)")
    for r_, c_ in zip(grid.r, grid.cost): ax.text(r_, c_ + 600, format(int(c_), ","), ha="center", fontsize=9, color=COL["BN"])
    ax.set_xticks(list(grid.r)); ax.set_xlabel("Cost ratio r (FN cost : FP cost)"); ax.set_ylabel("Total test cost = r x FN + FP"); ax.legend(frameon=False, loc="upper left")
    ax.set_title("Selection BN: test cost across the pre-specified cost-ratio grid")
    out[NAMES[2]] = finish(fig, [ax], "Grid r in {5, 10, 15, 20} fixed a priori; tau_r = 1/(1+r); r = 5 is primary. At r = 5 no cell reaches tau, so the BN equals accept-all. Test set, n = 30,000 (2,005 defaulters).")

    # 4 flip rates
    fig, ax = plt.subplots(figsize=(11, 6.5)); rs = sorted(va.r.unique()); x = np.arange(len(rs)); w = 0.36
    for k, v in enumerate(("Age", "Debt Ratio")):
        g = va[va.variable == v].sort_values("r"); f = g.flip_rate.values * 100
        ax.bar(x + (k - 0.5) * w, f, w, color=PAL[v], label=v, yerr=[f - g.flip_ci_lo.values * 100, g.flip_ci_hi.values * 100 - f], capsize=4)
        for xi, fi in zip(x + (k - 0.5) * w, f): ax.text(xi, fi + 0.9, "%.2f%%" % fi, ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(["r = %d\ntau = 1/%d" % (r, r + 1) for r in rs]); ax.set_ylabel("Decision flips when the variable is retracted (% of test rows)")
    ax.legend(frameon=False); ax.set_title("Selection BN, VoI Part A: how often retracting Age or Debt Ratio changes the decision")
    out[NAMES[3]] = finish(fig, [ax], "Flip = decision at tau_r differs between full evidence and evidence without the variable. 95%% bootstrap CIs (2,000 resamples, seed 42), %d test rows (2 rows with zero-probability retracted evidence excluded). At r = 5 no baseline posterior reaches tau, so there are 0 flips." % int(va.n_rows.iloc[0]))

    # 5 distributions
    order = list(rq2.sort_values("mean_S_exp", ascending=False).variable)
    data = [per[per.variable.map(short) == v].S_exp.dropna().to_numpy() for v in order]
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    ax.boxplot(data, showfliers=False, medianprops=dict(color="black", lw=1.5), widths=0.55)
    xs = np.arange(1, len(order) + 1)
    ax.scatter(xs, [d_.mean() for d_ in data], marker="D", color="tab:red", zorder=3, label="mean")
    ax.scatter(xs, [np.percentile(d_, 99) for d_ in data], marker="x", color="black", zorder=3, label="99th percentile")
    ax.set_yscale("symlog", linthresh=0.001); ax.set_ylim(0, 0.12)
    ax.set_xticks(xs); ax.set_xticklabels(["%s\n(n=%s)" % (v, format(len(d_), ",")) for v, d_ in zip(order, data)], fontsize=9)
    ax.set_ylabel("S_exp (symlog axis; linear below 0.001)"); ax.legend(loc="upper right", frameon=False)
    ax.set_title("Distribution of S_exp across test rows: the six selection variables")
    out[NAMES[4]] = finish(fig, [ax], "Income, Dependents, Real Estate Loans and Open Credit Lines are outside Default's Markov blanket {Age, Debt Ratio}: S_exp = 0 on every row (structural zero). Test rows, NaN-aware.")

    # 6 ROC
    fig, ax = plt.subplots(figsize=(8, 8))
    for k in ("LR", "RF", "BN"):
        fpr, tpr, _ = roc_curve(y, P[k]); ax.plot(fpr, tpr, color=COL[k], label="%s (AUC=%.4f)" % (k, roc_auc_score(y, P[k])))
    ax.plot([0, 1], [0, 1], color="gray", ls=":", label="Chance"); ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_title("ROC Curves, six selection variables"); ax.legend(loc="lower right", frameon=False)
    out[NAMES[5]] = finish(fig, [ax], "Test set, n = 30,000. LR (C=1, l2) and RF (500 trees, depth 10, min leaf 20) use all six selection variables; the BN's Default depends on Age and Debt Ratio only (25 distinct scores).")

    # 7 ranking
    t = rq2.sort_values("mean_S_exp", ascending=False).reset_index(drop=True); isb = t.in_blanket.values.astype(bool); yy = np.arange(len(t)); xm = float(t.mean_S_exp.max())
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    ax.barh(yy[isb], t.mean_S_exp[isb], color="tab:blue", height=0.6)
    for yi, m in zip(yy[isb], t.mean_S_exp[isb]): ax.text(m + 0.015 * xm, yi, "%.4f" % m, va="center", fontsize=9)
    ax.plot(np.zeros((~isb).sum()), yy[~isb], ls="none", marker="o", ms=6, markerfacecolor="white", markeredgecolor="dimgray", clip_on=False)
    ax.set_yticks(yy); ax.set_yticklabels(["%s  (n=%s)" % (v, format(int(k), ",")) for v, k in zip(t.variable, t.n_rows)]); ax.invert_yaxis()
    ax.text(0.07 * xm * 1.6, yy[~isb].mean(), "outside Default's Markov blanket:\nstructural zero (S_exp = 0 on every row)", va="center", color="dimgray", fontsize=10)
    ax.set_xlim(0, xm * 1.22); ax.set_xlabel("Mean S_exp (probability-weighted shift in P(Default=1))")
    ax.set_title("RQ2: one-way evidence sensitivity of the selection BN's P(Default=1)\nall 30,000 test rows")
    out[NAMES[6]] = finish(fig, [ax], "n = rows on which the variable is observed and S_exp is defined. Bootstrap 95% CIs (2,000 resamples, seed 42): Age 0.02683 to 0.02711; Debt Ratio 0.01206 to 0.01226.")

    for n, b in out.items():
        p = ss.check_path(os.path.join(R, F + n))
        with open(p, "xb") as f: f.write(b)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b).hexdigest()
        print("saved %-40s %7d bytes sha12 %s" % (n, len(b), hashlib.sha256(b).hexdigest()[:12]))
_c144()

In [ ]:
# Cell 145 (04_baselines_rq2.ipynb): redraw flip-rate figure without error bars -> sel_flip_rates_by_ratio_v2.png (exclusive create). No restart.
def _c145():
    import os, sys, io, hashlib, textwrap, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    rel = "selection/figures/sel_flip_rates_by_ratio_v2.png"
    ss.check_path(os.path.join(R, rel))
    va = pd.read_csv(R + "/selection/tables/sel_voi_partA_table.csv")
    PAL = {"Age": "#0072B2", "Debt Ratio": "#E69F00"}
    rs = sorted(va.r.unique()); x = np.arange(len(rs)); w = 0.36
    fig, ax = plt.subplots(figsize=(11, 6.5))
    for k, v in enumerate(("Age", "Debt Ratio")):
        g = va[va.variable == v].sort_values("r"); f = g.flip_rate.values * 100
        ax.bar(x + (k - 0.5) * w, f, w, color=PAL[v], label=v)
        for xi, fi in zip(x + (k - 0.5) * w, f):
            ax.text(xi, fi + 0.5, "%.2f%%" % fi, ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(["r = %d\ntau = 1/%d" % (r, r + 1) for r in rs])
    ax.set_ylabel("Decision flips when the variable is retracted (% of test rows)")
    ax.legend(frameon=False)
    ax.set_title("Selection BN, VoI Part A: how often retracting Age or Debt Ratio changes the decision")
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.tight_layout(rect=[0, 0.09, 1, 1])
    cap = ("Flip = decision at tau_r differs between full evidence and evidence without the variable. %d test rows (2 rows with "
           "zero-probability retracted evidence excluded). At r = 5 no baseline posterior reaches tau, so there are 0 flips. "
           "95%% bootstrap CIs are in sel_voi_partA_table.csv." % int(va.n_rows.iloc[0]))
    fig.text(0.01, 0.005, textwrap.fill(cap, 140), fontsize=8, va="bottom")
    b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig)
    p = ss.check_path(os.path.join(R, rel))
    with open(p, "xb") as f: f.write(b.getvalue())
    assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b.getvalue()).hexdigest()
    print("saved sel_flip_rates_by_ratio_v2.png | %d bytes | sha12 %s" % (len(b.getvalue()), hashlib.sha256(b.getvalue()).hexdigest()[:12]))
_c145()

In [ ]:
# Cell 146 (04_baselines_rq2.ipynb): Phase 5 manifest/git DRY RUN. Writes NOTHING. No restart.
def _c146():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    def git(*a):
        return subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    print("DRY_RUN = True")
    print("manifest rows now:", len(existing), "(expected 148) | ends with newline:", raw.endswith(b'\n'))
    tag = git("rev-list", "-n", "1", "full-variable-final")
    print("tag points to:", tag[:12], "| OK:", tag.startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())
    print("old flip-rate figure present:", os.path.exists(os.path.join(R, "selection/figures/sel_flip_rates_by_ratio.png")))

    rows = []
    for top in ("selection", "shared"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                if f.endswith(".pyc") or f == ".DS_Store":
                    continue
                p = os.path.join(dp, f); rel = os.path.relpath(p, R)
                if rel in existing:
                    continue
                rows.append((rel, os.path.getsize(p), sha(p)[:12], time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
    print("\nrows to append (%d):" % len(rows))
    for r in sorted(rows):
        print("  ", *r)
    print("expected manifest rows after:", len(existing) + len(rows))
    print("\nwould commit: MANIFEST.csv, shared/src/sensitivity.py, selection/notebooks/04_baselines_rq2.ipynb (data/pkl/csv/png/json are git-ignored by design)")
    print("DRY RUN ONLY: nothing written.")
_c146()

In [ ]:
# Cell 147 (04_baselines_rq2.ipynb): REAL RUN (approved). Manifest append (22 Phase 5 files) + NEW git commit. Tag never touched.
def _c147():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    NOTE = "added after Phase 5, 2026-10-15"
    NB = "selection/notebooks/04_baselines_rq2.ipynb"
    F = [("selection/figures/sel_calibration.png", "6780ede7067c"),
         ("selection/figures/sel_cost_vs_ratio.png", "1132fe43d5af"),
         ("selection/figures/sel_default_heatmap.png", "09c5a56ed65e"),
         ("selection/figures/sel_flip_rates_by_ratio_v2.png", "877da33d632f"),
         ("selection/figures/sel_roc_curves.png", "1706abdbbb81"),
         ("selection/figures/sel_rq2_sensitivity_distributions.png", "6b78ca714376"),
         ("selection/figures/sel_rq2_sensitivity_ranking.png", "b6cb27262f9a"),
         ("selection/models/sel_lr_grid.pkl", "a1272e482531"),
         ("selection/models/sel_rf_grid.pkl", "866022a5a306"),
         (NB, None),
         ("selection/predictions/sel_baseline_test_predictions.pkl", "0b300b55d67e"),
         ("selection/predictions/sel_rq2_per_row.pkl", "db1620f3cf51"),
         ("selection/tables/sel_additive_lbfgs_diagnostic.csv", "0d40874f056e"),
         ("selection/tables/sel_baselines_bootstrap.csv", "5ec0fcff835e"),
         ("selection/tables/sel_baselines_table.csv", "851d7c4585f4"),
         ("selection/tables/sel_cost_ratio_grid.csv", "85922df28ce3"),
         ("selection/tables/sel_em_vs_deletion.csv", "4bdc7b73f83f"),
         ("selection/tables/sel_em_vs_deletion_metrics.json", "69495bf9e32c"),
         ("selection/tables/sel_rq2_sensitivity_table.csv", "31d75e1ad138"),
         ("selection/tables/sel_voi_partA_table.csv", "e5d73d649998"),
         ("selection/tables/sel_voi_partB_table.csv", "89a7e71696bb"),
         ("shared/src/sensitivity.py", "e49b368ee726")]
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(a)} failed: {r.stderr.strip()[:300]}")
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw0 = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    assert raw0.endswith(b'\n') and len(existing) == 148, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"
    rows = []
    for rel, h12 in F:
        p = os.path.join(R, rel)
        assert rel not in existing and os.path.exists(p), f"unexpected state for {rel}: STOP"
        h = sha(p)
        if h12 is not None:
            assert h.startswith(h12), f"hash changed since dry run for {rel}: STOP"
        else:
            print("notebook hash recorded now: %s (dry run showed 8613571cdb1d)" % h[:12])
        role = "shared" if rel.startswith("shared/") else "selection"
        rows.append([rel, os.path.getsize(p), h, role, time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE])
    with open(mp, 'a', newline='') as fh:
        csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 170 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", "shared/src/sensitivity.py", NB)
    ident = []
    if not git("config", "user.name", check=False) or not git("config", "user.email", check=False):
        ident = ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 5: sensitivity.py, 04_baselines_rq2 notebook, baselines/EM-vs-deletion/RQ2/VoI outputs added to manifest")
    print("manifest rows now:", n, "(expected 170)")
    print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status:", git("status", "--porcelain").splitlines())
_c147()